# NLP System Design Lab: Intent Classification for Telecom

In this lab we build a complete intent classification system from scratch:
- Define intents and create labeled training data
- Preprocess text (lowercase, tokenize, remove stopwords)
- Vectorize with TF-IDF
- Train a LogisticRegression classifier
- Evaluate with a classification report and confusion matrix

This is a simplified version of what real telecom chatbots use for routing customer messages.

In [ ]:
!pip install -q nltk

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

## 1. Define Intents and Training Data

We define 6 intents with 10 example messages each (60 total).

In [ ]:
data = {
    'text': [
        # billing_inquiry (10)
        "how much is my current bill",
        "I want to know my balance",
        "when is my next payment due",
        "can you explain the charges on my bill",
        "why was I charged extra this month",
        "how do I pay my bill online",
        "is there a way to get a billing statement",
        "my bill seems too high this month",
        "what payment methods do you accept",
        "I already paid but it still shows balance",

        # technical_support (10)
        "my internet is not working",
        "the wifi keeps disconnecting",
        "internet speed is very slow",
        "I cannot connect to the network",
        "my router is blinking red light",
        "there is no signal on my phone",
        "the connection drops every few minutes",
        "why is my download speed so low",
        "modem is not turning on",
        "I am getting timeout errors on every website",

        # plan_change (10)
        "I want to upgrade my plan",
        "how can I switch to a cheaper plan",
        "what plans do you currently offer",
        "can I change my tariff to unlimited",
        "I need more data on my plan",
        "how to downgrade my subscription",
        "what is the difference between basic and premium plans",
        "I want to add international calling to my plan",
        "can I switch plans in the middle of the month",
        "show me available internet packages",

        # complaint (10)
        "I am very unhappy with your service",
        "this is the worst experience I have ever had",
        "I want to file a formal complaint",
        "your customer service is terrible",
        "I have been waiting for a technician for three days",
        "nobody is helping me solve my problem",
        "I am going to switch to another provider",
        "this is unacceptable and I want a refund",
        "I have called five times and nothing is fixed",
        "your company does not care about customers",

        # account_info (10)
        "what is my account number",
        "I need to update my address",
        "how do I change my password",
        "can I change the name on my account",
        "I forgot my login credentials",
        "how to reset my account password",
        "what email is linked to my account",
        "I want to add another user to my account",
        "how do I check my data usage",
        "please update my phone number on file",

        # other (10)
        "where is your nearest office",
        "what are your working hours",
        "do you have a mobile app",
        "hello I just wanted to say thank you",
        "is there a loyalty program for long time customers",
        "can I get a student discount",
        "what promotions are running right now",
        "do you sponsor any events",
        "I saw your ad on TV and I am interested",
        "how long has your company been in business"
    ],
    'intent': (
        ['billing_inquiry'] * 10 +
        ['technical_support'] * 10 +
        ['plan_change'] * 10 +
        ['complaint'] * 10 +
        ['account_info'] * 10 +
        ['other'] * 10
    )
}

df = pd.DataFrame(data)
print(f"Total samples: {len(df)}")
print(f"\nIntent distribution:")
print(df['intent'].value_counts())

## 2. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    """Lowercase, tokenize, remove stopwords and non-alpha tokens."""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)

# Show a few examples
df[['text', 'clean_text']].head(8)

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['intent'],
    test_size=0.25,
    random_state=42,
    stratify=df['intent']
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 4. TF-IDF + Logistic Regression Pipeline

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=500, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

## 5. Confusion Matrix

In [ ]:
labels = sorted(df['intent'].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Intent Classification - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Test with New Messages

In [ ]:
test_messages = [
    "my internet has been down since morning",
    "I want to see my latest invoice",
    "can I get a better plan with more data",
    "this is ridiculous I want to speak to a manager",
    "how do I reset my login password",
    "do you have any stores near the city center"
]

clean_test = [preprocess(msg) for msg in test_messages]
predictions = pipeline.predict(clean_test)

for msg, pred in zip(test_messages, predictions):
    print(f"  [{pred:>20}]  {msg}")

## 7. Your Turn

### TODO 1: Add 5 more messages per intent, retrain, and check if accuracy improves

Add your new messages to the dataframe, re-run the pipeline, and compare the classification report.

In [ ]:
# TODO: Create a list of 30 new messages (5 per intent)
# new_texts = [
#     # billing_inquiry (5 more)
#     "...",
#     # technical_support (5 more)
#     "...",
#     # plan_change (5 more)
#     "...",
#     # complaint (5 more)
#     "...",
#     # account_info (5 more)
#     "...",
#     # other (5 more)
#     "...",
# ]
# new_intents = ['billing_inquiry']*5 + ['technical_support']*5 + ...
#
# df_new = pd.concat([df, pd.DataFrame({'text': new_texts, 'intent': new_intents})], ignore_index=True)
# df_new['clean_text'] = df_new['text'].apply(preprocess)
#
# X_train2, X_test2, y_train2, y_test2 = train_test_split(
#     df_new['clean_text'], df_new['intent'], test_size=0.25, random_state=42, stratify=df_new['intent']
# )
# pipeline2 = Pipeline([('tfidf', TfidfVectorizer(max_features=500, ngram_range=(1,2))),
#                       ('clf', LogisticRegression(max_iter=1000, random_state=42))])
# pipeline2.fit(X_train2, y_train2)
# print(classification_report(y_test2, pipeline2.predict(X_test2)))


### TODO 2: How would you handle messages that don't fit any intent?

Write your proposal below. Consider:
- Should "other" be the catch-all? What are the downsides?
- Could you use a confidence threshold to flag uncertain predictions?
- What happens in production when new intent types emerge?

**Your proposal:**

...

In [ ]:
# Bonus: Check prediction probabilities to see how confident the model is
# This can help identify messages that should be flagged for human review

ambiguous_messages = [
    "I need help with my account and also my bill is wrong",
    "hello",
    "asdfghjkl"
]

clean_ambiguous = [preprocess(msg) for msg in ambiguous_messages]
probas = pipeline.predict_proba(clean_ambiguous)

for msg, proba in zip(ambiguous_messages, probas):
    pred_idx = np.argmax(proba)
    confidence = proba[pred_idx]
    pred_label = pipeline.classes_[pred_idx]
    print(f"  Message: \"{msg}\"")
    print(f"  Predicted: {pred_label} (confidence: {confidence:.2%})")
    print()